# ARGUS — Colab Master Notebook

Smart glasses AI for the visually impaired. Trains on Colab, deploys to Jetson Orin Nano Super.

**Before starting:** `Runtime → Change runtime type → GPU (A100 or T4)`

**Every session:** Run cells 1 → 2 → 3 → 4 → 5 in order. Keep this browser tab open.

| Cell | What it does | Time |
|------|-------------|------|
| 1 | GPU check + Mount Google Drive | ~10 s |
| 2 | Clone or pull latest code | ~10 s |
| 3 | Install all Python packages | ~3 min |
| 4 | Download datasets & model weights | **first run ~30 min**, then ~10 s |
| 5 | Train all models end-to-end | **hours** — leave running |
| 6 | Status check | ~5 s |
| 7 | Re-run one specific model | optional |

> **Never use the Colab terminal for long-running tasks.** Notebook cells survive reconnects; the terminal does not.

---
## Cell 1 — Mount Google Drive & Check GPU

Run this first every session. It mounts your Drive and creates the ARGUS folder structure.

In [ ]:
import torch, os
from pathlib import Path

# GPU check
assert torch.cuda.is_available(), 'No GPU — go to Runtime → Change runtime type → GPU'
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create ARGUS folder structure (safe to re-run)
BASE = Path('/content/drive/MyDrive/ARGUS')
for d in [
    BASE/'datasets/ade20k',         BASE/'datasets/nyuv2',
    BASE/'datasets/coco',           BASE/'datasets/custom_stereo',
    BASE/'datasets/wakeword/positive',
    BASE/'models/depth',            BASE/'models/segmentation',
    BASE/'models/detection',        BASE/'models/privacy/easyocr',
    BASE/'models/speech/piper',     BASE/'models/speech/whisper',
    BASE/'models/llm',              BASE/'checkpoints',
    BASE/'logs',                    BASE/'exports',
]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Drive: {BASE} — ready')

---
## Cell 2 — Clone / Pull Repository

Clones the repo on first run of a session. Pulls the latest code on subsequent runs.

In [ ]:
import os
REPO = '/content/Argus'

if not os.path.exists(REPO):
    print('Cloning ARGUS repo...')
    !git clone https://github.com/meteorboyF/Argus.git {REPO}
else:
    print('Pulling latest code...')
    !cd {REPO} && git pull --ff-only origin main

# Verify key files
for f in ['download_datasets.py', 'run_argus_pipeline.py', 'src/privacy_filter.py']:
    exists = os.path.exists(os.path.join(REPO, f))
    print(f'  {"OK" if exists else "MISSING"}: {f}')

---
## Cell 3 — Install All Packages

Installs every package needed for all 7 pipeline stages.  
Takes ~3 minutes. Must run once per session.

In [ ]:
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

print('[1/6] Core ML...')
pip('torch', 'torchvision', 'torchaudio')

print('[2/6] Computer vision...')
pip('opencv-python-headless', 'Pillow', 'albumentations',
    'timm', 'einops', 'kornia', 'scikit-image')

print('[3/6] Detection, segmentation & LLM...')
pip('ultralytics', 'transformers', 'datasets', 'accelerate',
    'peft', 'bitsandbytes', 'trl')

print('[4/6] Privacy filter + ONNX export...')
pip('insightface', 'easyocr', 'onnxruntime-gpu', 'onnx', 'onnxscript')

print('[5/6] Speech...')
pip('openai-whisper', 'SpeechRecognition', 'piper-tts',
    'openwakeword', 'soundfile', 'sounddevice')

print('[6/6] Utilities...')
pip('huggingface_hub', 'gdown', 'wandb', 'tensorboard',
    'tqdm', 'scipy', 'scikit-learn', 'pandas',
    'matplotlib', 'pyyaml', 'psutil', 'ninja')

print('\n✅ All packages installed.')

---
## Cell 4 — Download Datasets & Model Weights

**First run:** saves ~24 GB to Google Drive. Takes ~30 min total.  
**All future runs:** skips everything already on Drive. Takes ~10 seconds.

What gets downloaded:
- ADE20K ~900 MB, NYUv2 ~2.8 GB, COCO 2017 ~18 GB
- RAFT-Stereo weights ~30 MB
- SCRFD buffalo_s face detector ~90 MB
- EasyOCR English models ~100 MB
- Phi-3.5 Mini GGUF ~2.7 GB
- Piper TTS ~65 MB

> **EasyOCR:** downloads from GitHub and shows a slow progress bar. It may appear stuck at a low percentage for 2–3 minutes. **This is normal — do not interrupt.**
>
> **COCO train2017:** 18 GB, takes ~8 min at Colab speeds. **Do not interrupt.** If interrupted, re-running this cell resumes automatically.

In [ ]:
!python /content/Argus/download_datasets.py

---
## Cell 5 — Train All Models

Runs the full pipeline in sequence:
1. **Stereo Depth** — RAFT-Stereo fine-tune
2. **Semantic Segmentation** — SegFormer-B2 (ADE20K + NYUv2)
3. **Object Detection** — YOLOv8-small (COCO 2017)
4. **Privacy Filter** — face + text blurring verification
5. **Speech Pipeline** — Whisper STT + Piper TTS + Wake Word
6. **LLM Setup** — Phi-3.5 Mini + optional LoRA fine-tune
7. **Integration Test** — full pipeline smoke test

**Checkpoints save to Google Drive automatically.** If the session disconnects mid-training, re-run this cell and it picks up where it left off.

> Keep this browser tab open and visible. Training takes several hours on A100.

In [ ]:
!python /content/Argus/run_argus_pipeline.py

---
## Cell 6 — Status Check

Run any time to see what's ready on Google Drive.

In [ ]:
from pathlib import Path
import subprocess

BASE   = Path('/content/drive/MyDrive/ARGUS')
MODELS = BASE / 'models'
DS     = BASE / 'datasets'

def sz(p):
    p = Path(p)
    if not p.exists(): return 'missing'
    if p.is_file(): return f'{p.stat().st_size/1e6:.0f} MB'
    total = sum(f.stat().st_size for f in p.rglob('*') if f.is_file())
    return f'{total/1e9:.1f} GB'

print('=' * 58)
print('  ARGUS — Drive Status')
print('=' * 58)

sections = [
    ('Datasets',[
        ('ADE20K',         DS/'ade20k'/'ade20k_done.flag'),
        ('NYUv2',          DS/'nyuv2'/'nyu_depth_v2_labeled.mat'),
        ('COCO 2017',      DS/'coco'/'coco_done.flag'),
    ]),
    ('Base Model Weights', [
        ('RAFT-Stereo',    MODELS/'depth'/'raftstereo-realtime.pth'),
        ('SCRFD buffalo_s',MODELS/'privacy'/'models'/'buffalo_s'/'det_500m.onnx'),
        ('EasyOCR',        MODELS/'privacy'/'easyocr'/'easyocr_done.flag'),
        ('Phi-3.5 GGUF',   MODELS/'llm'/'Phi-3.5-mini-instruct-Q4_K_M.gguf'),
        ('Piper TTS',      MODELS/'speech'/'piper'/'en_US-lessac-medium.onnx'),
    ]),
    ('Trained Models', [
        ('Depth model',    MODELS/'depth'/'latest.pth'),
        ('SegFormer',      MODELS/'segmentation'/'latest.pth'),
        ('YOLOv8',         MODELS/'detection'/'latest.pt'),
        ('LoRA adapter',   MODELS/'segmentation'/'phi35_lora'/'DONE'),
    ]),
]

for section_name, items in sections:
    print(f'\n  {section_name}')
    for label, path in items:
        exists = Path(path).exists()
        icon   = '' if exists else ''
        size   = sz(path)
        print(f'    {icon}  {label:<22} {size}')

r = subprocess.run(['du', '-sh', str(BASE)], capture_output=True, text=True)
total = r.stdout.split()[0] if r.returncode == 0 else 'unknown'
print(f'\n  Total Drive usage: {total}')
print('=' * 58)

---
## Cell 7 — Re-run One Specific Model (optional)

Only use this to debug or re-train a single step. Cell 5 runs all of them automatically.

In [ ]:
NB      = '/content/Argus/notebooks'
TIMEOUT = '--ExecutePreprocessor.timeout=-1'

# Uncomment ONE line below and run this cell:

# !jupyter nbconvert --to notebook --execute {NB}/01_stereo_depth.ipynb     --inplace {TIMEOUT}
# !jupyter nbconvert --to notebook --execute {NB}/02_segmentation.ipynb     --inplace {TIMEOUT}
# !jupyter nbconvert --to notebook --execute {NB}/03_object_detection.ipynb --inplace {TIMEOUT}
# !jupyter nbconvert --to notebook --execute {NB}/04_privacy_filter.ipynb   --inplace {TIMEOUT}
# !jupyter nbconvert --to notebook --execute {NB}/05_speech_pipeline.ipynb  --inplace {TIMEOUT}
# !jupyter nbconvert --to notebook --execute {NB}/06_llm_setup.ipynb        --inplace {TIMEOUT}
# !jupyter nbconvert --to notebook --execute {NB}/07_integration_test.ipynb --inplace {TIMEOUT}

print('Uncomment the model you want to re-run above, then run this cell.')